# Executing Quality Checks locally

This notebook runs the Reportnet 3 Quality Checks of the *River Basin Districts and Competent Authorities* dataflow against the deliverables produced by the prefill notebook, without submitting anything to Reportnet 3.

The QCs are stored in the `qc` schema of `wise_rbdsuca.duckdb` (built from the `.sql` files under `sql/qc/`), and they address the reported data through the same schema names Reportnet 3 uses:

| schema | resolves to |
| --- | --- |
| `descriptive_reporting` | the exported SQLite database |
| `spatial_reporting` | the exported GeoPackage, read through `ST_Read` so geometries are typed |
| `reference` | the reference datasets published in the data lake |

> `wise_rbdsuca.duckdb` ships precompiled - no build step needed. Run the prefill notebook first so `output/<CC>/` has the SQLite + GeoPackage this notebook attaches.

In [5]:
%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


## 1. Initial setup

In [24]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "wise_local_qc.py").exists())
sys.path.insert(0, str(ROOT))

import ipywidgets as widgets

import wise_local_qc as wq

con = wq.connect(ROOT / "wise_rbdsuca.duckdb")
wq.current_parameters(con)

{'country_code': 'AT', 'cycle_year': '2022', 'reference_cycle_year': '2022'}

## 2. Country selection

`Reference cycle` is the cycle the reference QCs compare the submission against. Choosing a cycle other than the reported one is a quick way to make the reference QCs produce findings.

The dropdown only lists countries that have already been prefilled and appear under `output/`. If a country is missing, run the **Prefill** notebook (`0_prefill/Prefill_RiverBasinDistrictsAndCompetentAuthorities.ipynb`) for that country first.

In [25]:
countries = wq.available_countries(ROOT / "output")
default_country = next((code for _, code in countries if code == "AT"), countries[0][1] if countries else None)
countries_selection = widgets.Dropdown(options=countries, value=default_country, description="Country:")
cycle_selection = widgets.Dropdown(options=["2022", "2016", "2010"], value="2022", description="Cycle:")
reference_selection = widgets.Dropdown(options=["2022", "2016", "2010"], value="2022", description="Reference:")
widgets.VBox([countries_selection, cycle_selection, reference_selection])

In [26]:
wq.set_parameters(
    con,
    country_code=countries_selection.value,
    cycle_year=cycle_selection.value,
    reference_cycle_year=reference_selection.value,
)

output_dir = ROOT / "output" / countries_selection.value
sqlite_path = output_dir / "RiverBasinDistrictsAndCompetentAuthorities.sqlite"
geopackage_path = output_dir / "RiverBasinDistrict.gpkg"

wq.attach_reporting(con, sqlite_path, geopackage_path)
wq.current_parameters(con)

{'country_code': 'SE', 'cycle_year': '2022', 'reference_cycle_year': '2016'}

## 3. Available tables

The reference views come from the catalog, the reporting views from the files just attached.

In [27]:
con.sql(
    """
    SELECT schema_name, view_name AS table_name
    FROM duckdb_views()
    WHERE schema_name IN ('reference', 'descriptive_reporting', 'spatial_reporting')
    ORDER BY schema_name, view_name
    """
).df()

,schema_name,table_name
0,descriptive_reporting,CompetentAuthority
1,descriptive_reporting,RiverBasinDistrictCompetentAuthority
2,reference,Country
3,reference,RiverBasinDistrictWFD
4,spatial_reporting,RiverBasinDistrict


In [28]:
con.sql("SELECT * EXCLUDE (geom) FROM spatial_reporting.RiverBasinDistrict").df()

,inspireIdLocalId,inspireIdNamespace,inspireIdVersionId,thematicIdIdentifier,thematicIdIdentifierScheme,beginLifespanVersion,endLifespanVersion,predecessorsIdentifier,predecessorsIdentifierScheme,successorsIdentifier,...,nameLanguage,designationPeriodBegin,designationPeriodEnd,zoneType,specialisedZoneType,legalBasisName,legalBasisLink,legalBasisLevel,link,record_id
0,SE1_INT,SE.SMHI.AM.ManagementRestrictionOrRegulationZone,None,SE1_INT,euRBDCode,2021-12-02,None,"SE1,SE1103,SE1104,SE1TO",euRBDCode,None,...,swe,2022-09-05,None,riverBasinDistrict,internationalRiverBasinDistrict,None,None,None,None,c77ee5cd-2e30-992f-c366-8b301fe6228a
1,SE2_INT,SE.SMHI.AM.ManagementRestrictionOrRegulationZone,None,SE2_INT,euRBDCode,2021-11-30,None,"SE1102,SE2",euRBDCode,None,...,swe,2022-09-05,None,riverBasinDistrict,internationalRiverBasinDistrict,None,None,None,None,a2f4384e-6aba-686d-6aa2-6d77f1d52370
2,SE3,SE.SMHI.AM.ManagementRestrictionOrRegulationZone,None,SE3,euRBDCode,2021-12-02,None,None,None,None,...,swe,2009-12-22,None,riverBasinDistrict,nationalRiverBasinDistrict,None,None,None,None,63d40b39-4403-9c6e-2e06-18b767104d1b
3,SE4,SE.SMHI.AM.ManagementRestrictionOrRegulationZone,None,SE4,euRBDCode,2021-11-30,None,None,None,None,...,swe,2009-12-22,None,riverBasinDistrict,nationalRiverBasinDistrict,None,None,None,None,bd7f5a8e-4f99-24b2-947e-1005e6218028
4,SE5_INT,SE.SMHI.AM.ManagementRestrictionOrRegulationZone,None,SE5_INT,euRBDCode,2021-12-01,None,"SE5,SE5101",euRBDCode,None,...,swe,2022-09-05,None,riverBasinDistrict,internationalRiverBasinDistrict,None,None,None,None,ab29a96c-3079-23b8-40f6-40f2b7083ac3


## 4. Available Quality Checks

In [29]:
wq.list_qcs(con)

,code,table_name,error_level,description
0,R003,SPATIAL_RiverBasinDistrict,ERROR,endLifespanVersion must be later than beginLif...
1,R011,SPATIAL_RiverBasinDistrict,ERROR,Creation objects must not report predecessors.
2,R014,SPATIAL_RiverBasinDistrict,ERROR,predecessorsIdentifierScheme must be reported ...
3,R015,SPATIAL_RiverBasinDistrict,ERROR,Each predecessor identifier must match an exis...
4,R016,SPATIAL_RiverBasinDistrict,ERROR,Deletion and noChange objects must not report ...
5,R022,SPATIAL_RiverBasinDistrict,ERROR,Change objects must report inspireIdVersionId.
6,R026,SPATIAL_RiverBasinDistrict,ERROR,changeCode and splitting objects must report e...
7,R027,SPATIAL_RiverBasinDistrict,ERROR,Predecessor identifiers must not be repeated w...
8,R028,SPATIAL_RiverBasinDistrict,ERROR,Objects with wiseEvolutionType creation or noC...
9,R029,SPATIAL_RiverBasinDistrict,ERROR,A successorsIdentifier must correspond to an e...


## 5. Execute the Quality Checks

A QC returns the offending records, so an empty result means the check passed.

In [30]:
summary, results = wq.run_qcs(con)
summary

,code,error_level,status,records,description,message
0,R003,ERROR,PASSED,0,endLifespanVersion must be later than beginLif...,
1,R011,ERROR,PASSED,0,Creation objects must not report predecessors.,
2,R014,ERROR,PASSED,0,predecessorsIdentifierScheme must be reported ...,
3,R015,ERROR,ERROR,0,Each predecessor identifier must match an exis...,BinderException: Binder Error: No function mat...
4,R016,ERROR,PASSED,0,Deletion and noChange objects must not report ...,
5,R022,ERROR,PASSED,0,Change objects must report inspireIdVersionId.,
6,R026,ERROR,PASSED,0,changeCode and splitting objects must report e...,
7,R027,ERROR,ERROR,0,Predecessor identifiers must not be repeated w...,InvalidInputException: Invalid Input Error: Un...
8,R028,ERROR,PASSED,0,Objects with wiseEvolutionType creation or noC...,
9,R029,ERROR,ERROR,0,A successorsIdentifier must correspond to an e...,InvalidInputException: Invalid Input Error: Un...


In [31]:
wq.write_qc_results(results, output_dir / "qc_results")

[]


## 6. AI-assisted interpretation of the failures

For every QC with status `FAILED` or `ERROR`, ask an LLM to explain what happened, its
likely root cause and recommended actions - built from the QC's own description, its
SQL and either a sample of the offending records (`FAILED`) or the exception message
(`ERROR`), so no separate documentation has to be maintained per QC.

Requires a provider to be configured through environment variables (see
`ai_explain.py`'s docstring): `AI_EXPLAIN_PROVIDER` defaults to `openai`, which only
needs `OPENAI_API_KEY` in a `.env` file (see `.env.example`).

By default every failing/erroring QC is sent (`ALL`). Use the selector below to pick
only specific QC codes instead - hold Ctrl/Cmd to select several.

In [32]:
failing_qc_codes = summary.loc[summary["status"].isin(["FAILED", "ERROR"]), "code"].tolist()
qc_ai_selection = widgets.SelectMultiple(
    options=["ALL"] + failing_qc_codes,
    value=("ALL",),
    description="QCs:",
    rows=min(len(failing_qc_codes) + 1, 8),
)
qc_ai_selection

SelectMultiple(description='QCs:', index=(0,), options=('ALL', 'R015', 'R027', 'R029', 'R032', 'R035', 'R046',…

In [33]:
import ai_explain
from IPython.display import Markdown, display

selected_qc_codes = None if "ALL" in qc_ai_selection.value else list(qc_ai_selection.value)
display(Markdown(ai_explain.explain_failures(con, summary, results, codes=selected_qc_codes)))

### R027 (ERROR - execution failed)

**What failed**: The SQL query failed due to an invalid regular expression option 'A' in the STR_SPLIT_REGEX function.
**Root cause**: The `STR_SPLIT_REGEX` function in the `FLATTEN` expression.
**Likely reasons**:
* The `predecessorsIdentifier` column contains a value that is not a comma-separated list.
* The `predecessorsIdentifier` column contains a value with a different separator than a comma.
* The `predecessorsIdentifier` column is not of a string type.
**Recommended actions**:
* Check the data type and format of the `predecessorsIdentifier` column.
* Verify that the `predecessorsIdentifier` column contains only comma-separated lists.
* Adjust the regular expression option in the `STR_SPLIT_REGEX` function to a valid option.

---

### R035 (ERROR - execution failed)

**What failed**: The SQL query failed due to an invalid regular expression option 'A' in the STR_SPLIT_REGEX function.
**Root cause**: The `STR_SPLIT_REGEX` function in the line `TRIM(FLATTEN(STR_SPLIT_REGEX(predecessorsIdentifier, ',', 'ALL'))) AS onePredecessorIdentifier,`.
**Likely reasons**:
* The `STR_SPLIT_REGEX` function is not supported in the current DuckDB version.
* The regular expression option 'A' is not a valid option for the `STR_SPLIT_REGEX` function.
* The `STR_SPLIT_REGEX` function is being used with an incorrect syntax.
**Recommended actions**:
* Replace the `STR_SPLIT_REGEX` function with a supported function, such as `STR_SPLIT`.
* Check the documentation for the correct syntax and options for the `STR_SPLIT_REGEX` function.
* Consider using a different approach to split the string, such as using the `FLATTEN` function with a simple split character.

---

### T230 (ERROR - execution failed)

**What failed**: A duplicate CTE name "vSRIDs" was encountered in the SQL query.
**Root cause**: The second CTE named "vSRIDs" in the SQL query.
**Likely reasons**:
* The table or CTE was renamed in the dataflow, but not updated in the QC.
* The QC was copied and pasted, resulting in duplicate CTE names.
* The table or CTE was not exported for this dataflow, but the QC still references it.
**Recommended actions**:
* Check the meta.datasets for the dataflow to confirm the table or CTE names.
* Adjust the SQL query to use unique CTE names.
* Confirm that this QC applies to this dataflow and that the table or CTE is correctly referenced.

---

### V040 (ERROR - execution failed)

**What failed**: The execution of the QC V040 failed due to an invalid Perl operator in the regular expression.
**Root cause**: The regular expression in the `regexp` column of the `Data` CTE.
**Likely reasons**:
* The `getvariable('country_code')` function is not defined in the schema.
* The `getvariable` function is not supported in DuckDB.
* The regular expression pattern is incorrect or incomplete.
**Recommended actions**:
* Check the schema to see if the `getvariable` function is defined and if it returns a valid country code.
* Replace the `getvariable` function with a valid country code or a column that contains the country code.
* Adjust the regular expression pattern to use a valid Perl operator or a different pattern that achieves the same result.

## 7. Debug a single Quality Check

Print the SQL as stored in the catalog, then run it on its own. To iterate on a check, edit the corresponding file under `sql/qc/`, re-run `python build_catalog.py` and re-run the cells below.

In [16]:
qc_selection = widgets.Dropdown(options=wq.list_qcs(con)["code"].tolist(), description="QC:")
qc_selection

Dropdown(description='QC:', options=('R003', 'R011', 'R014', 'R015', 'R016', 'R022', 'R026', 'R027', 'R028', '…

In [17]:
print(wq.qc_sql(con, qc_selection.value))

WITH BothDates AS (
  SELECT
    record_id,
    CAST(beginLifespanVersion AS DATE) AS beginDate,
    CAST(endLifespanVersion AS DATE) AS endDate
  FROM spatial_reporting.RiverBasinDistrict
  /* Avoids values not reported */
  WHERE
    COALESCE(beginLifespanVersion, '') <> ''
    AND COALESCE(endLifespanVersion, '') <> ''
)
SELECT
  record_id,
  beginDate,
  endDate
FROM BothDates
WHERE
  NOT beginDate IS NULL AND NOT endDate IS NULL AND endDate <= beginDate


In [18]:
wq.run_qc(con, qc_selection.value)

,record_id,beginDate,endDate


## 8. Optional: see the checks fire

Prefilled data is consistent by construction, so every QC passes. The cell below writes a deliberately broken copy of the GeoPackage - one overlapping River Basin District and one shifted `designationPeriodBegin` - and points `spatial_reporting` at it, which makes S016 and RF012_WFD fail.

Re-run the *Country selection* cell to go back to the untouched deliverable.

In [19]:
wq.build_sandbox(con, geopackage_path, output_dir / "sandbox" / "RiverBasinDistrict.gpkg")
summary, results = wq.run_qcs(con)
summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,code,error_level,status,records,description,message
0,R003,ERROR,PASSED,0,endLifespanVersion must be later than beginLif...,
1,R011,ERROR,PASSED,0,Creation objects must not report predecessors.,
2,R014,ERROR,PASSED,0,predecessorsIdentifierScheme must be reported ...,
3,R015,ERROR,ERROR,0,Each predecessor identifier must match an exis...,BinderException: Binder Error: No function mat...
4,R016,ERROR,PASSED,0,Deletion and noChange objects must not report ...,
5,R022,ERROR,PASSED,0,Change objects must report inspireIdVersionId.,
6,R026,ERROR,PASSED,0,changeCode and splitting objects must report e...,
7,R027,ERROR,ERROR,0,Predecessor identifiers must not be repeated w...,InvalidInputException: Invalid Input Error: Un...
8,R028,ERROR,PASSED,0,Objects with wiseEvolutionType creation or noC...,
9,R029,ERROR,ERROR,0,A successorsIdentifier must correspond to an e...,InvalidInputException: Invalid Input Error: Un...


In [20]:
results["S016"]

,record_id,thematicIdIdentifier,overlapsWith
0,b76c08d3-94d3-a620-bb1c-f4fbb2597d0d,ES010,XX9999


In [34]:
con.close()